### Report purpose

This notebook converts the raw master table into a modeling-ready dataset for taxi fare prediction. The central objective is to improve data quality without introducing target leakage.

The work in this notebook follows four principles: remove corrupted or implausible trips, keep only information that would be available at prediction time, enrich location context with TLC zone metadata, and export a dataset that can be reused consistently across models.


### Load the source tables

The notebook dynamically resolves the project root, then loads the consolidated trip table and the official taxi zone lookup used for spatial enrichment.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path().resolve()
project_root = next(p for p in [cwd] + list(cwd.parents) if (p / "data").exists())

path_output = project_root / "data" / "procesed"
master_path = path_output / "master_2019_1M_per_month.parquet"
zone_path = path_output / "taxi_zone_lookup.csv"
model_path = path_output / "taxi_2019_modeling_ready.parquet"

df = pd.read_parquet(master_path)
zones = pd.read_csv(zone_path)


In [3]:
print("master shape:", df.shape)
print("zone lookup shape:", zones.shape)
df.head()

master shape: (12000000, 21)
zone lookup shape: (265, 4)


,DOLocationID,PULocationID,RatecodeID,VendorID,congestion_surcharge,extra,fare_amount,improvement_surcharge,mta_tax,passenger_count,...,store_and_fwd_flag,tip_amount,tolls_amount,total_amount,tpep_dropoff_datetime,tpep_pickup_datetime,trip_distance,year,month,source_file
0,234,164,1.0,2.0,NaN,0.0,4.5,0.3,0.5,1.0,...,N,1.06,0.0,6.36,2019-01-09 09:31:43,2019-01-09 09:28:06,0.52,2019,1,yellow_tripdata_2019-01.csv
1,230,100,1.0,2.0,NaN,0.0,7.0,0.3,0.5,1.0,...,N,1.95,0.0,9.75,2019-01-02 07:37:09,2019-01-02 07:29:10,1.15,2019,1,yellow_tripdata_2019-01.csv
2,162,140,1.0,2.0,NaN,1.0,10.5,0.3,0.5,1.0,...,N,0.00,0.0,12.30,2019-01-07 16:06:42,2019-01-07 15:55:27,2.44,2019,1,yellow_tripdata_2019-01.csv
3,239,151,1.0,1.0,NaN,0.0,5.5,0.3,0.5,1.0,...,N,1.25,0.0,7.55,2019-01-09 06:56:05,2019-01-09 06:52:41,1.20,2019,1,yellow_tripdata_2019-01.csv
4,260,140,1.0,1.0,NaN,0.0,20.0,0.3,0.5,1.0,...,N,0.00,0.0,20.80,2019-01-17 09:14:37,2019-01-17 08:50:24,4.60,2019,1,yellow_tripdata_2019-01.csv


### Remove structurally incomplete records

The first cleaning step removes the small set of rows where essential trip metadata is jointly missing. These records are not reliable enough to support downstream validation or modeling.


In [4]:
core_vendor_cols = ["VendorID", "store_and_fwd_flag", "passenger_count"]
mask_block_missing = df[core_vendor_cols].isna().all(axis=1)

print("rows fully missing core vendor block:", mask_block_missing.sum())
print("share:", round(mask_block_missing.mean(), 6))

df = df.loc[~mask_block_missing].copy()

rows fully missing core vendor block: 37009
share: 0.003084


### Parse timestamps for validation

Pickup and dropoff timestamps are parsed so the notebook can validate trip chronology and derive pickup-time features. The timestamps are not kept as raw modeling fields because they are mainly a preparation aid at this stage.


In [5]:
for c in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")

df = df.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"]).copy()
df["pickup_month_num"] = df["tpep_pickup_datetime"].dt.month
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_weekday"] = df["tpep_pickup_datetime"].dt.weekday


### Filter invalid and implausible trips

This section removes observations that would distort the target or describe impossible travel behavior, including non-positive fares, zero-distance trips, invalid durations, and unrealistic speeds. This is one of the most important controls in the project because model quality depends heavily on target quality.


In [6]:
df["duration_min"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60
df["speed_mph"] = df["trip_distance"] / (df["duration_min"] / 60)

valid_trip_mask = (
    (df["fare_amount"] > 0) &
    (df["trip_distance"] > 0) &
    (df["duration_min"] > 0) &
    (df["duration_min"] < 6 * 60) &
    (df["speed_mph"] > 0) &
    (df["speed_mph"] < 120)
)

print("rows kept after trip validation:", int(valid_trip_mask.sum()))
print("share kept:", round(valid_trip_mask.mean(), 4))

df = df.loc[valid_trip_mask].copy()

rows kept after trip validation: 11792502
share kept: 0.9857


### Clean passenger count

Impossible passenger counts are converted to missing values rather than treated as factual observations. That lets the downstream pipeline impute them instead of learning patterns from impossible inputs.


In [7]:
df["passenger_count_missing"] = (~df["passenger_count"].between(1, 6)).astype(int)
df["passenger_count"] = df["passenger_count"].where(df["passenger_count"].between(1, 6))

df[["passenger_count", "passenger_count_missing"]].describe(include="all")

,passenger_count,passenger_count_missing
count,1.158036e+07,1.179250e+07
mean,1.593319e+00,1.798982e-02
std,1.200990e+00,1.329142e-01
min,1.000000e+00,0.000000e+00
25%,1.000000e+00,0.000000e+00
50%,1.000000e+00,0.000000e+00
75%,2.000000e+00,0.000000e+00
max,6.000000e+00,1.000000e+00


### Engineer pickup-time features

These variables describe the pickup moment in a leakage-safe way. They capture calendar and time-of-day structure while avoiding realized trip information that would only be known after the ride ends.


In [8]:
df["is_weekend"] = (df["pickup_weekday"] >= 5).astype(int)
df["hour_sin"] = np.sin(2 * np.pi * df["pickup_hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["pickup_hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["pickup_month_num"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["pickup_month_num"] / 12)

df["is_rush_hour"] = df["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)
df["is_night"] = df["pickup_hour"].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
df["is_peak_daytime"] = ((df["pickup_hour"] >= 11) & (df["pickup_hour"] <= 18)).astype(int)


### Engineer distance-based features

Trip distance is one of the strongest fare drivers, so the notebook adds a small set of nonlinear distance transforms and interactions. The goal is to give the models more expressive signal without making the feature set hard to interpret.


In [9]:
df["log_trip_distance"] = np.log1p(df["trip_distance"])
df["trip_distance_sq"] = df["trip_distance"] ** 2
df["distance_x_rush"] = df["trip_distance"] * df["is_rush_hour"]
df["distance_x_weekend"] = df["trip_distance"] * df["is_weekend"]


### Enrich trips with zone metadata

The TLC zone lookup is joined to pickup and dropoff locations so the final table contains interpretable borough and service-zone information. This adds spatial context without relying on post-trip monetary variables.


In [10]:
pu_lookup = zones.rename(
    columns={
        "LocationID": "PULocationID",
        "Borough": "pickup_borough",
        "Zone": "pickup_zone",
        "service_zone": "pickup_service_zone",
    }
)

do_lookup = zones.rename(
    columns={
        "LocationID": "DOLocationID",
        "Borough": "dropoff_borough",
        "Zone": "dropoff_zone",
        "service_zone": "dropoff_service_zone",
    }
)

df = df.merge(
    pu_lookup[["PULocationID", "pickup_borough", "pickup_zone", "pickup_service_zone"]],
    on="PULocationID",
    how="left",
)
df = df.merge(
    do_lookup[["DOLocationID", "dropoff_borough", "dropoff_zone", "dropoff_service_zone"]],
    on="DOLocationID",
    how="left",
)

airport_zones = {"Newark Airport", "JFK Airport", "LaGuardia Airport"}
df["is_airport_pickup"] = df["pickup_zone"].isin(airport_zones).astype(int)
df["is_airport_dropoff"] = df["dropoff_zone"].isin(airport_zones).astype(int)
df["same_borough_trip"] = (df["pickup_borough"] == df["dropoff_borough"]).astype(int)
df["manhattan_trip"] = (
    (df["pickup_borough"] == "Manhattan") |
    (df["dropoff_borough"] == "Manhattan")
).astype(int)


### Assemble the final modeling table

At this point the notebook drops validation-only columns and any fields that would leak post-trip information into the model.

`pickup_month_num` is retained only as a chronological split helper for evaluation. It should not be used as a raw predictor because the cyclical month encoding already captures that signal in a more appropriate form.


In [11]:
cols_to_drop = [
    "VendorID",
    "RatecodeID",
    "payment_type",
    "store_and_fwd_flag",
    "source_file",
    "year",
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "duration_min",
    "speed_mph",
    "pickup_hour",
    "pickup_zone",
    "dropoff_zone",
]

df_model = df.drop(columns=cols_to_drop, errors="ignore").copy()

ordered_cols = [
    "fare_amount",
    "pickup_month_num",
    "trip_distance",
    "log_trip_distance",
    "trip_distance_sq",
    "distance_x_rush",
    "distance_x_weekend",
    "passenger_count",
    "passenger_count_missing",
    "pickup_weekday",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "is_rush_hour",
    "is_night",
    "is_peak_daytime",
    "PULocationID",
    "DOLocationID",
    "pickup_borough",
    "dropoff_borough",
    "pickup_service_zone",
    "dropoff_service_zone",
    "is_airport_pickup",
    "is_airport_dropoff",
    "same_borough_trip",
    "manhattan_trip",
]

df_model = df_model[ordered_cols]
df_model.to_parquet(model_path, index=False)


In [12]:
print("final model shape:", df_model.shape)
print("saved to:", model_path)
print("\nfinal columns:\n")
print(df_model.columns.tolist())

df_model.head()

final model shape: (11792502, 28)
saved to: C:\Users\leodo\Desktop\NYC-Taxi-ML\data\procesed\taxi_2019_modeling_ready.parquet

final columns:

['fare_amount', 'pickup_month_num', 'trip_distance', 'log_trip_distance', 'trip_distance_sq', 'distance_x_rush', 'distance_x_weekend', 'passenger_count', 'passenger_count_missing', 'pickup_weekday', 'is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_rush_hour', 'is_night', 'is_peak_daytime', 'PULocationID', 'DOLocationID', 'pickup_borough', 'dropoff_borough', 'pickup_service_zone', 'dropoff_service_zone', 'is_airport_pickup', 'is_airport_dropoff', 'same_borough_trip', 'manhattan_trip']


,fare_amount,pickup_month_num,trip_distance,log_trip_distance,trip_distance_sq,distance_x_rush,distance_x_weekend,passenger_count,passenger_count_missing,pickup_weekday,...,PULocationID,DOLocationID,pickup_borough,dropoff_borough,pickup_service_zone,dropoff_service_zone,is_airport_pickup,is_airport_dropoff,same_borough_trip,manhattan_trip
0,4.5,1,0.52,0.418710,0.2704,0.52,0.0,1.0,0,2,...,164,234,Manhattan,Manhattan,Yellow Zone,Yellow Zone,0,0,1,1
1,7.0,1,1.15,0.765468,1.3225,1.15,0.0,1.0,0,2,...,100,230,Manhattan,Manhattan,Yellow Zone,Yellow Zone,0,0,1,1
2,10.5,1,2.44,1.235471,5.9536,0.00,0.0,1.0,0,0,...,140,162,Manhattan,Manhattan,Yellow Zone,Yellow Zone,0,0,1,1
3,5.5,1,1.20,0.788457,1.4400,0.00,0.0,1.0,0,2,...,151,239,Manhattan,Manhattan,Yellow Zone,Yellow Zone,0,0,1,1
4,20.0,1,4.60,1.722767,21.1600,4.60,0.0,1.0,0,3,...,140,260,Manhattan,Queens,Yellow Zone,Boro Zone,0,0,0,1


### Chapter summary

The exported dataset is stricter and more modeling-safe than the raw master table in three ways: invalid targets and impossible trips are removed, leakage-prone post-trip fields are excluded, and locations are enriched with interpretable geographic context.

That combination makes the next modeling notebooks easier to trust, easier to compare, and easier for a new reader to understand.
